# Qwen3.5-4B Telecom RCA: SFT + Held-Out Evaluation

This Colab notebook fine-tunes `unsloth/Qwen3.5-4B` with 16-bit LoRA on the synthetic telecom RCA reasoning trajectories, evaluates held-out loss during training, and measures generated-label accuracy on all 864 official validation questions.

Key rules:

- The synthetic reasoning is trained in Qwen3.5's native `<think>...</think>` format.
- The validation set contains only known final labels. No validation reasoning is invented.
- Validation examples are never optimizer inputs.
- Qwen3.5 4-bit QLoRA is intentionally disabled because Unsloth currently recommends 16-bit LoRA for this model family.
- GRPO is a separate stage and must reuse this exact base model, tokenizer/chat template, and response contract.


In [1]:
!git clone https://github.com/joshsalako/telelogs.git

Cloning into 'telelogs'...
remote: Enumerating objects: 314, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 314 (delta 43), reused 48 (delta 23), pack-reused 242 (from 2)
Receiving objects: 100% (314/314), 209.28 MiB | 24.91 MiB/s, done.
Resolving deltas: 100% (145/145), done.
Updating files: 100% (93/93), done.


In [2]:
import os
import sys

print("Python:", sys.executable)

# Install uv using the notebook's actual Python.
# Use only one -q, not -qqq.
!{sys.executable} -m pip install -q --upgrade uv

# Make every uv command install into the notebook environment.
os.environ["UV_SYSTEM_PYTHON"] = "1"

Python: /usr/bin/python3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 80.2 MB/s eta 0:00:00:00:0100:01


## 1. Install the Qwen3.5-compatible Unsloth stack

Use a GPU runtime: **Runtime → Change runtime type → T4 GPU**. The first run compiles Qwen3.5's hybrid-model kernels and can take several minutes.


In [3]:
!uv pip install --upgrade \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo.git" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth.git" \
    bitsandbytes \
    "xformers==0.0.32.post2" \
    datasets \
    pandas

Using Python 3.12.13 environment at: /usr
Resolved 102 packages in 15.95s                                      
Prepared 43 packages in 54.88s                                           
Uninstalled 34 packages in 1.55s
Installed 43 packages in 500ms                              
 - aiohttp==3.14.1
 + aiohttp==3.14.3
 - annotated-types==0.7.0
 + annotated-types==0.8.0
 + bitsandbytes==0.50.0
 - certifi==2026.6.17
 + certifi==2026.7.22
 + cut-cross-entropy==25.1.1
 - datasets==4.0.0
 + datasets==4.3.0
 - dill==0.3.8
 + dill==0.4.0
 - filelock==3.29.7
 + filelock==3.32.0
 - fsspec==2025.3.0
 + fsspec==2025.9.0
 + hf-transfer==0.1.9
 - hf-xet==1.5.1
 + hf-xet==1.5.2
 - huggingface-hub==1.23.0
 + huggingface-hub==1.25.1
 + msgspec==0.21.1
 - numpy==2.0.2
 + numpy==2.5.1
 - nvidia-cudnn-cu12==9.19.0.56
 + nvidia-cudnn-cu12==9.10.2.21
 - nvidia-nccl-cu12==2.28.9
 + nvidia-nccl-cu12==2.27.3
 - pandas==2.2.2
 + pandas==3.0.5
 - pillow==11.3.0
 + pillow==12.3.0
 - protobuf==5.29.6
 + protobuf==7

In [4]:
!uv pip install --upgrade --no-deps \
    "transformers==5.2.0" \
    "tokenizers>=0.22.0,<=0.23.0" \
    "trl==0.22.2" \
    "torchao>=0.16.0"

Using Python 3.12.13 environment at: /usr
Resolved 4 packages in 89ms                                          
Prepared 2 packages in 662ms                                             
Uninstalled 2 packages in 78ms
Installed 2 packages in 46ms                                
 - transformers==5.5.0
 + transformers==5.2.0
 - trl==0.24.0
 + trl==0.22.2


In [1]:
from importlib.metadata import version, PackageNotFoundError

packages = [
    "torch",
    "transformers",
    "trl",
    "unsloth",
    "unsloth_zoo",
    "tokenizers",
    "xformers",
    "bitsandbytes",
    "torchao",
]

for package in packages:
    try:
        print(f"{package:25s}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package:25s}: NOT INSTALLED")

torch                    : 2.8.0
transformers             : 5.2.0
trl                      : 0.22.2
unsloth                  : 2026.7.5
unsloth_zoo              : 2026.7.6
tokenizers               : 0.22.2
xformers                 : 0.0.32.post2
bitsandbytes             : 0.50.0
torchao                  : 0.17.0


## 2. Configuration, dynamic paths, and resume policy

The notebook searches the current directory, each `data/` and `synthesis/`
subdirectory, `/content`, `/root`, their `telelogs` clones, and equivalent
Google Drive `MyDrive` and `MyDrive/telelogs` locations. It resolves
`sft_train_data.jsonl`, `sft_validation_data.jsonl`, and any starting adapter
independently and prints their absolute paths.

`RESUME=True` is the default. It strictly requires `qwen35_4b_sft_lora`,
validates and loads every LoRA tensor, and uses full Trainer-state recovery only
when all four Trainer state files exist. Otherwise it performs a weights-only
continuation with a fresh optimizer and scheduler. Resumed checkpoints and the
final adapter are written to `qwen35_4b_sft_outputs_resumed` and
`qwen35_4b_sft_lora_resumed`, leaving the source adapter untouched. Set
`RESUME=False` to initialize a new LoRA over the base model.

In [2]:
import json
import os
import random
import re
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset

SEED = 42
MODEL_NAME = "unsloth/Qwen3.5-4B"
MAX_SEQ_LENGTH = 8192
LORA_RANK = 16
LORA_ALPHA = 16
RESUME = True
OUTPUT_DIR = "qwen35_4b_sft_outputs_resumed" if RESUME else "qwen35_4b_sft_outputs"
ADAPTER_DIR = "qwen35_4b_sft_lora_resumed" if RESUME else "qwen35_4b_sft_lora"
GENERATION_MAX_NEW_TOKENS = 1024

# Set to a positive integer for a quick smoke test. Keep None for the official
# all-864-question evaluation.
EVAL_LIMIT = 10

SYSTEM_PROMPT = (
    "You are a senior telecom root-cause analysis engineer. Analyze the supplied "
    "drive-test and engineering evidence carefully. Follow the candidate identifiers "
    "defined in the user prompt and finish with exactly one selected identifier "
    "enclosed in \\boxed{}."
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "A CUDA GPU runtime is required."
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())


def search_roots():
    """Return ordered, deduplicated roots supported by every training notebook."""
    cwd = Path.cwd()
    bases = [
        cwd,
        Path("/content"),
        Path("/content/telelogs"),
        Path("/root"),
        Path("/root/telelogs"),
        Path("/content/drive/MyDrive"),
        Path("/content/drive/MyDrive/telelogs"),
    ]
    roots = []
    seen = set()
    for base in bases:
        for root in (base, base / "data", base / "synthesis"):
            absolute = root.expanduser().resolve()
            key = os.path.normcase(str(absolute))
            if key not in seen:
                seen.add(key)
                roots.append(absolute)
    return roots


def locate_path(name, *, expected, required=True):
    """Locate a named file or directory with deterministic first-match precedence."""
    if expected not in {"file", "directory"}:
        raise ValueError("expected must be 'file' or 'directory'")
    requested = Path(name).expanduser()
    candidates = (
        [requested]
        if requested.is_absolute()
        else [root / requested for root in search_roots()]
    )
    unique_candidates = []
    seen = set()
    for candidate in candidates:
        absolute = candidate.resolve()
        key = os.path.normcase(str(absolute))
        if key not in seen:
            seen.add(key)
            unique_candidates.append(absolute)
    predicate = Path.is_file if expected == "file" else Path.is_dir
    for candidate in unique_candidates:
        if predicate(candidate):
            print(f"Resolved {expected} {name!r}: {candidate}")
            return candidate
    if required:
        locations = "\n  ".join(str(path) for path in unique_candidates)
        raise FileNotFoundError(
            f"Could not find required {expected} {name!r}. Searched:\n  {locations}"
        )
    print(f"Optional {expected} {name!r} was not found.")
    return None


TRAINER_STATE_FILES = (
    "trainer_state.json",
    "optimizer.pt",
    "scheduler.pt",
    "rng_state.pth",
)


def validate_adapter(adapter_dir):
    """Validate adapter files and the configuration shared by SFT and GRPO."""
    adapter_dir = Path(adapter_dir).resolve()
    model_file = adapter_dir / "adapter_model.safetensors"
    config_file = adapter_dir / "adapter_config.json"
    missing = [
        str(path.name) for path in (model_file, config_file) if not path.is_file()
    ]
    if missing:
        raise FileNotFoundError(
            f"Adapter {adapter_dir} is missing required files: {', '.join(missing)}"
        )
    with config_file.open(encoding="utf-8") as handle:
        config = json.load(handle)
    expected = {
        "base_model_name_or_path": MODEL_NAME,
        "r": LORA_RANK,
        "lora_alpha": LORA_ALPHA,
    }
    mismatches = {
        key: {"expected": value, "actual": config.get(key)}
        for key, value in expected.items()
        if config.get(key) != value
    }
    if mismatches:
        raise ValueError(
            f"Incompatible adapter configuration at {config_file}: {mismatches}"
        )
    print(
        "Validated adapter:",
        adapter_dir,
        f"(base={MODEL_NAME}, r={LORA_RANK}, alpha={LORA_ALPHA})",
    )
    return adapter_dir, model_file


def select_resume_mode(resume, adapter_dir):
    if not resume:
        return "fresh", None
    missing_state = [
        name for name in TRAINER_STATE_FILES if not (adapter_dir / name).is_file()
    ]
    if missing_state:
        print(
            "Trainer state is incomplete; using loaded adapter weights with a fresh "
            f"optimizer/scheduler. Missing: {', '.join(missing_state)}"
        )
        return "weights-only resume", None
    return "full Trainer-state resume", adapter_dir


TRAIN_PATH = locate_path("sft_train_data.jsonl", expected="file")
VALIDATION_PATH = locate_path("sft_validation_data.jsonl", expected="file")
STARTING_ADAPTER_PATH = (
    locate_path("qwen35_4b_sft_lora", expected="directory") if RESUME else None
)
ADAPTER_FILE = None
if STARTING_ADAPTER_PATH is not None:
    STARTING_ADAPTER_PATH, ADAPTER_FILE = validate_adapter(STARTING_ADAPTER_PATH)
RESUME_MODE, TRAINER_RESUME_PATH = select_resume_mode(RESUME, STARTING_ADAPTER_PATH)
print("Resume mode:", RESUME_MODE)
print("Trainer output directory:", Path(OUTPUT_DIR).resolve())
print("Final adapter directory:", Path(ADAPTER_DIR).resolve())

GPU: Tesla T4
BF16 supported: True
Resolved file 'sft_train_data.jsonl': /content/telelogs/synthesis/sft_train_data.jsonl
Resolved file 'sft_validation_data.jsonl': /content/telelogs/data/sft_validation_data.jsonl
Resolved directory 'qwen35_4b_sft_lora': /content/telelogs/qwen35_4b_sft_lora
Validated adapter: /content/telelogs/qwen35_4b_sft_lora (base=unsloth/Qwen3.5-4B, r=16, alpha=16)
Trainer state is incomplete; using loaded adapter weights with a fresh optimizer/scheduler. Missing: trainer_state.json, optimizer.pt, scheduler.pt, rng_state.pth
Resume mode: weights-only resume
Trainer output directory: /content/qwen35_4b_sft_outputs_resumed
Final adapter directory: /content/qwen35_4b_sft_lora_resumed


## 3. Load Qwen3.5-4B, attach LoRA, and optionally restore its weights

`fast_inference=False` is deliberate. Vision parameters remain frozen because
this task is text-only. A fresh run initializes LoRA over the base model; a
resumed run requires a fully compatible adapter and loads every expected tensor
before training.

In [3]:
from unsloth import FastLanguageModel
import torch

# Enable meta-data support to bypass potential torch.nonzero() errors on certain kernels
torch.fx.experimental._config.meta_nonzero_assume_all_nonzero = True

# Load model in 4-bit to fit within T4 GPU memory (16GB)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
    fast_inference=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)


def normalize_adapter_key(key):
    key = key.replace(".default.", ".")
    prefixes = ("base_model.", "model.", "language_model.")
    changed = True
    while changed:
        changed = False
        for prefix in prefixes:
            if key.startswith(prefix):
                key = key[len(prefix) :]
                changed = True
    return key


def load_all_lora_weights(model, adapter_file):
    """Load the selected adapter and reject missing, extra, or mismatched LoRA tensors."""
    from safetensors.torch import load_file as safe_load

    saved_state = safe_load(str(adapter_file))
    normalized_saved = {}
    for key, tensor in saved_state.items():
        if "lora_" not in key:
            continue
        normalized = normalize_adapter_key(key)
        if normalized in normalized_saved:
            raise RuntimeError(f"Duplicate normalized adapter key: {normalized}")
        normalized_saved[normalized] = (key, tensor)

    model_state = model.state_dict()
    expected = {
        normalize_adapter_key(key): (key, tensor)
        for key, tensor in model_state.items()
        if "lora_" in key
    }
    missing = sorted(set(expected) - set(normalized_saved))
    unexpected = sorted(set(normalized_saved) - set(expected))
    if missing or unexpected:
        raise RuntimeError(
            "Adapter tensor keys do not exactly match the attached LoRA. "
            f"Missing={missing[:10]}; unexpected={unexpected[:10]}"
        )

    for normalized, (model_key, destination) in expected.items():
        saved_key, source = normalized_saved[normalized]
        if source.shape != destination.shape:
            raise RuntimeError(
                f"Shape mismatch for {saved_key} -> {model_key}: "
                f"saved={tuple(source.shape)}, expected={tuple(destination.shape)}"
            )
        destination.data.copy_(source.to(destination.device, destination.dtype))
    print(
        f"Loaded all {len(expected)} expected LoRA tensors from "
        f"{Path(adapter_file).resolve()}"
    )


if ADAPTER_FILE is not None:
    load_all_lora_weights(model, ADAPTER_FILE)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Loaded 4-bit QLoRA base:", MODEL_NAME)
print("Starting adapter:", STARTING_ADAPTER_PATH)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth/_gpu_init.py:226: UserWarning: torchcodec 0.11.0+cu128 is incompatible with torch 2.8.0+cu128; install a matching build with `pip install 'torchcodec>=0.7,<0.8.0'`.
  disable_torchcodec_if_broken()


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.5: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
Loaded all 256 expected LoRA tensors from /content/telelogs/qwen35_4b_sft_lora/adapter_model.safetensors
Loaded 4-bit QLoRA base: unsloth/Qwen3.5-4B
Starting adapter: /content/telelogs/qwen35_4b_sft_lora


## 4. Validate and format the datasets

Training responses are converted from their existing four-section form into:

```text
<think>
...synthetic reasoning...
</think>

\boxed{R#}
```

Validation responses remain answer-only (`\boxed{C#}`). They support completion-only held-out loss without fabricating reasoning that is absent from the source data.


In [11]:
TRAIN_BOX_RE = re.compile(r"\\boxed\{(R[1-8])\}[.!?;:]*\s*$")
VALID_RESPONSE_RE = re.compile(r"\\boxed\{(C[1-8])\}")


def load_jsonl(path):
    records = []
    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"{path}:{line_number}: {error}") from error
            if not isinstance(item, dict):
                raise ValueError(f"{path}:{line_number} must be a JSON object")
            records.append(item)
    return records


def format_training_response(response, location):
    if not isinstance(response, str) or not response.strip():
        raise ValueError(f"{location}: response must be non-empty")
    matches = list(TRAIN_BOX_RE.finditer(response))
    all_boxes = re.findall(r"\\boxed\{R[1-8]\}", response)
    if len(matches) != 1 or len(all_boxes) != 1:
        raise ValueError(f"{location}: expected exactly one terminal boxed R1-R8 label")
    match = matches[0]
    reasoning = response[: match.start()].rstrip()
    if not reasoning:
        raise ValueError(f"{location}: reasoning is empty")
    return f"<think>\n{reasoning}\n</think>\n\n\\boxed{{{match.group(1)}}}"


def conversation(question, assistant_response):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
        {"role": "assistant", "content": assistant_response},
    ]


raw_train = load_jsonl(TRAIN_PATH)
raw_validation = load_jsonl(VALIDATION_PATH)
# assert len(raw_train) == 1941, f"Expected 1,941 SFT examples; got {len(raw_train)}"
# assert len(raw_validation) == 864, (
#     f"Expected 864 validation examples; got {len(raw_validation)}"
# )

train_rows = []
train_questions = set()
for index, item in enumerate(raw_train):
    if set(item) != {"question", "response"}:
        raise ValueError(f"train row {index}: expected only question and response")
    question = item["question"]
    if (
        not isinstance(question, str)
        or not question.strip()
        or question in train_questions
    ):
        raise ValueError(f"train row {index}: empty or duplicate question")
    response = format_training_response(item["response"], f"train row {index}")
    train_rows.append({"messages": conversation(question, response)})
    train_questions.add(question)

validation_rows = []
validation_questions = set()
validation_labels = Counter()
for index, item in enumerate(raw_validation):
    if set(item) != {"id", "question", "response"}:
        raise ValueError(f"validation row {index}: expected id, question, and response")
    identifier = item["id"]
    question = item["question"]
    response = item["response"]
    match = VALID_RESPONSE_RE.fullmatch(response)
    if not identifier or not question or not match:
        raise ValueError(f"validation row {index}: malformed id, question, or response")
    if question in validation_questions or question in train_questions:
        raise ValueError(
            f"validation row {index}: duplicate or train/validation overlap"
        )
    label = match.group(1)
    validation_rows.append(
        {
            "id": identifier,
            "question": question,
            "target": label,
            "messages": conversation(question, response),
        }
    )
    validation_questions.add(question)
    validation_labels[label] += 1

expected_balance = Counter({f"C{i}": 108 for i in range(1, 9)})
assert validation_labels == expected_balance, validation_labels
print(
    f"Validated {len(train_rows):,} SFT and {len(validation_rows):,} held-out examples"
)
print("Validation balance:", dict(sorted(validation_labels.items())))
print("Example formatted response tail:")
print(train_rows[0]["messages"][-1]["content"][-400:])

Validated 2,254 SFT and 864 held-out examples
Validation balance: {'C1': 108, 'C2': 108, 'C3': 108, 'C4': 108, 'C5': 108, 'C6': 108, 'C7': 108, 'C8': 108}
Example formatted response tail:
evice switches cells three times in four seconds, triggering continuous connection reconfiguration and radio interruption that suppresses throughput below 600 Mbps. All other potential causes are mathematically or empirically excluded based on colocation verification, distance thresholds, tilt/beamwidth relationships, PCI modulo checks, speed limits, and RB scheduling metrics.
</think>

\boxed{R1}


In [12]:
def render_manual(messages):
    # Manually construct the prompt to avoid chat template image-parsing side effects
    res = ""
    for m in messages:
        res += f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n"
    return res


train_text_rows = [{"text": render_manual(row["messages"])} for row in train_rows]
validation_text_rows = [
    {"text": render_manual(row["messages"])} for row in validation_rows
]
train_dataset = Dataset.from_list(train_text_rows)
validation_dataset = Dataset.from_list(validation_text_rows)


def token_length(text):
    # Access the tokenizer within the processor to avoid multimodal logic
    return len(tokenizer.tokenizer.encode(text, add_special_tokens=False))


train_lengths = [token_length(row["text"]) for row in train_text_rows]
validation_lengths = [token_length(row["text"]) for row in validation_text_rows]
max_observed = max(train_lengths + validation_lengths)

if max_observed > MAX_SEQ_LENGTH:
    raise ValueError(
        f"Longest example is {max_observed} tokens, exceeding MAX_SEQ_LENGTH."
    )

for name, lengths in (("train", train_lengths), ("validation", validation_lengths)):
    print(
        f"{name}: min={min(lengths)}, median={int(np.median(lengths))}, p95={int(np.percentile(lengths, 95))}, max={max(lengths)} tokens"
    )

probe = train_text_rows[0]["text"]
assert "<|im_start|>assistant\n" in probe
assert "<think>\n" in probe and "\\boxed{" in probe

train: min=2698, median=3324, p95=3779, max=4278 tokens
validation: min=2116, median=2439, p95=2765, max=2941 tokens


## 5. Supervised fine-tuning

The trainer masks every token before the assistant response. Consequently, validation loss is computed on the gold boxed answer rather than on the long question text. Packing is disabled so examples and masks cannot cross sequence boundaries.


In [13]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only
from transformers import DataCollatorForSeq2Seq
import gc

# Clear VRAM before initialization
torch.cuda.empty_cache()
gc.collect()

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    dataset_num_proc=1,
    packing=False,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=2e-4,
    warmup_steps=15,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=SEED,
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer.tokenizer, model=model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer.tokenizer,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    args=training_args,
    data_collator=data_collator,
)

# Mask prompt so model only learns from the <think> and \boxed{} parts
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/2254 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/864 [00:00<?, ? examples/s]

Map:   0%|          | 0/2254 [00:00<?, ? examples/s]

Map:   0%|          | 0/864 [00:00<?, ? examples/s]

In [ ]:
!nvidia-smi --query-gpu=timestamp,name,utilization.gpu,utilization.memory,memory.used,power.draw --format=csv -l 1

In [14]:
if TRAINER_RESUME_PATH is None:
    trainer_stats = trainer.train()
else:
    trainer_stats = trainer.train(resume_from_checkpoint=str(TRAINER_RESUME_PATH))

print(trainer_stats)
print("Resume mode:", RESUME_MODE)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best held-out loss:", trainer.state.best_metric)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,254 | Num Epochs = 5 | Total steps = 705
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 21,233,664 of 4,560,499,200 (0.47% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss


: 

: 

: 

In [13]:
torch.cuda.empty_cache()
import gc

gc.collect()

0

## 6. Full generated-label evaluation

This is the primary held-out metric. The model receives only the system prompt and question, generates its own reasoning and final answer, and receives credit only when the final non-whitespace text is a boxed `C1`–`C8` label.

Running all 864 long prompts on a T4 can take substantial time. Set `EVAL_LIMIT` in Section 2 only for debugging; leave it as `None` for the official result.


In [17]:
EVAL_LIMIT = None

In [ ]:
import re
import torch
from collections import defaultdict
from tqdm.auto import tqdm

# Enable Unsloth inference mode
FastLanguageModel.for_inference(model)
model.eval()

# Your tokenizer appears to be a processor/wrapper.
text_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer

# Required for batched generation with decoder-only models.
text_tokenizer.padding_side = "left"

# Some causal LMs do not define a separate padding token.
if text_tokenizer.pad_token_id is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token

# Keep wrapper settings synchronized when possible.
if hasattr(tokenizer, "padding_side"):
    tokenizer.padding_side = "left"

if hasattr(tokenizer, "pad_token_id") and tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# Keep model generation settings synchronized.
model.generation_config.pad_token_id = text_tokenizer.pad_token_id
model.generation_config.eos_token_id = text_tokenizer.eos_token_id

print("Padding side:", text_tokenizer.padding_side)
print("Pad token:", text_tokenizer.pad_token)
print("Pad token ID:", text_tokenizer.pad_token_id)
print("EOS token ID:", text_tokenizer.eos_token_id)


STRICT_FINAL_BOX_RE = re.compile(r"\\boxed\s*\{\s*(C[1-8])\s*\}")

evaluation_records = (
    validation_rows if EVAL_LIMIT is None else validation_rows[:EVAL_LIMIT]
)

predictions = []

BATCH_SIZE = 16

for i in tqdm(
    range(0, len(evaluation_records), BATCH_SIZE),
    desc="Batched Validation",
):
    batch = evaluation_records[i : i + BATCH_SIZE]

    prompts = []

    for row in batch:
        prompt_messages = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": row["question"],
            },
        ]

        prompt = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )

        prompts.append(prompt)

    # Left-padded batched tokenization
    inputs = text_tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        add_special_tokens=False,
    ).to(model.device)

    # All sequences have this padded input width.
    prompt_width = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=GENERATION_MAX_NEW_TOKENS,
            use_cache=True,
            do_sample=False,
            pad_token_id=text_tokenizer.pad_token_id,
            eos_token_id=text_tokenizer.eos_token_id,
        )

    # Remove the entire padded prompt portion.
    generated_ids = output_ids[:, prompt_width:]

    completions = text_tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )

    for row, completion in zip(batch, completions):
        completion = completion.strip()

        # Use the final boxed label if the response contains more than one.
        matches = STRICT_FINAL_BOX_RE.findall(completion)
        prediction = matches[-1] if matches else None

        predictions.append(
            {
                "id": row["id"],
                "target": row["target"],
                "prediction": prediction,
                "correct": prediction == row["target"],
                "format_valid": prediction is not None,
                "completion": completion,
            }
        )

correct = sum(record["correct"] for record in predictions)
total = len(predictions)

accuracy = correct / total if total else 0.0

print(f"Exact generated-label accuracy: {accuracy:.2%} ({correct}/{total})")

Padding side: left
Pad token: <|vision_pad|>
Pad token ID: 248055
EOS token ID: 248046


Batched Validation:   0%|          | 0/54 [00:00<?, ?it/s]

In [ ]:
import json
import pandas as pd

prediction_frame = pd.DataFrame(predictions)

labels = [f"C{i}" for i in range(1, 9)]

# Calculate summary metrics here so they always exist
total = len(prediction_frame)
correct = int(prediction_frame["correct"].sum())
accuracy = correct / total if total > 0 else 0.0

format_valid = int(prediction_frame["format_valid"].sum())
format_valid_rate = format_valid / total if total > 0 else 0.0

confusion = pd.crosstab(
    prediction_frame["target"],
    prediction_frame["prediction"].fillna("MALFORMED"),
    dropna=False,
).reindex(
    index=labels,
    columns=labels + ["MALFORMED"],
    fill_value=0,
)

# Reindex ensures every C1-C8 class appears, even if a class has no examples
per_class = (
    prediction_frame.groupby("target", sort=True)["correct"]
    .agg(["sum", "count", "mean"])
    .rename(
        columns={
            "sum": "correct",
            "mean": "accuracy",
        }
    )
    .reindex(labels)
)

# Replace missing counts with zero
per_class["correct"] = per_class["correct"].fillna(0).astype(int)
per_class["count"] = per_class["count"].fillna(0).astype(int)
per_class["accuracy"] = per_class["accuracy"].fillna(0.0)

display(per_class)
display(confusion)

print(f"Accuracy: {accuracy:.2%} ({correct}/{total})")
print(f"Valid output format: {format_valid_rate:.2%} ({format_valid}/{total})")

metrics = {
    "model": MODEL_NAME,
    "adapter": ADAPTER_DIR,
    "questions": total,
    "full_validation_set": EVAL_LIMIT is None,
    "correct": correct,
    "accuracy": accuracy,
    "format_valid": format_valid,
    "format_valid_rate": format_valid_rate,
    "per_class": {
        label: {
            "correct": int(per_class.loc[label, "correct"]),
            "count": int(per_class.loc[label, "count"]),
            "accuracy": float(per_class.loc[label, "accuracy"]),
        }
        for label in labels
    },
    "confusion_matrix": {
        target: {
            predicted: int(confusion.loc[target, predicted])
            for predicted in confusion.columns
        }
        for target in confusion.index
    },
}

with open(
    "validation_predictions.jsonl",
    "w",
    encoding="utf-8",
) as handle:
    for row in predictions:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")

with open(
    "validation_metrics.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        metrics,
        handle,
        ensure_ascii=False,
        indent=2,
    )

print("Saved validation_predictions.jsonl and validation_metrics.json")

## 7. Save the adapter and evaluation artifacts

The resulting directory is a LoRA adapter, not a merged model. Later GRPO must load it over `unsloth/Qwen3.5-4B` with `fast_inference=False` and preserve the native `<think>...</think>\n\n\boxed{...}` contract. Do not reuse a Qwen3-4B-Base GRPO notebook or its custom `<start_working_out>` tags unchanged.


In [ ]:
import shutil

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

with open(Path(ADAPTER_DIR) / "training_handoff.json", "w", encoding="utf-8") as handle:
    json.dump(
        {
            "base_model": MODEL_NAME,
            "resume_requested": RESUME,
            "resume_mode": RESUME_MODE,
            "starting_adapter": (
                str(STARTING_ADAPTER_PATH.resolve())
                if STARTING_ADAPTER_PATH is not None
                else None
            ),
            "max_seq_length": MAX_SEQ_LENGTH,
            "load_in_4bit": False,
            "load_in_16bit": True,
            "fast_inference": False,
            "response_contract": "<think>...</think>\\n\\n\\boxed{candidate}",
            "grpo_note": (
                "Load this adapter over the same Qwen3.5 base and tokenizer. "
                "Keep native thinking tags and reward the strict final boxed label."
            ),
        },
        handle,
        indent=2,
    )

adapter_archive = shutil.make_archive(ADAPTER_DIR, "zip", root_dir=ADAPTER_DIR)
print("Adapter archive:", adapter_archive)
print("Evaluation files: validation_predictions.jsonl, validation_metrics.json")

try:
    from google.colab import files

    files.download(adapter_archive)
    files.download("validation_metrics.json")
    files.download("validation_predictions.jsonl")
except ImportError:
    print("Not running in Colab; artifacts remain in the current directory.")